**Option 1: End-to-End Fine-Tuning**
- All parts (Model A, B, and fusion layer) are trained together.
- Allows the pretrained models to adapt to each other.

**Option 2: Frozen Backbone + Train Fusion Layer**
- Freeze Model A and B to preserve their pretrained knowledge.
- Only train the fusion logic and final classifier.
- Useful if you have limited data or want faster training.

______________________________________________________ **Example model setup** ___________________________________________________________

# Architecture

In [1]:
import torch
import torch.nn as nn

class EnsembleModel(nn.Module):
    def __init__(self, model_a, model_b, fusion='concat'):
        super().__init__()
        self.model_a = model_a
        self.model_b = model_b
        self.fusion = fusion  # 'concat', 'add', or custom logic

        # Example: Assume model_a and model_b output vectors of size 768
        fusion_output_size = 768 * 2 if fusion == 'concat' else 768

        self.classifier = nn.Linear(fusion_output_size, num_classes)

    def forward(self, x):
        out_a = self.model_a(x)  # Shape: (batch_size, 768)
        out_b = self.model_b(x)  # Shape: (batch_size, 768)

        if self.fusion == 'concat':
            fused = torch.cat([out_a, out_b], dim=1)
        elif self.fusion == 'add':
            fused = out_a + out_b
        else:
            raise ValueError("Unsupported fusion method")

        return self.classifier(fused)

# Training Loop

In [ ]:
model = EnsembleModel(pretrained_a, pretrained_b, fusion='concat')
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
loss_fn = nn.CrossEntropyLoss()

for x_batch, y_batch in dataloader:
    logits = model(x_batch)
    loss = loss_fn(logits, y_batch)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

# Freezing Pretrained Submodels

In [ ]:
for param in model.model_a.parameters():
    param.requires_grad = False

for param in model.model_b.parameters():
    param.requires_grad = False

# Tips and notes on Inputs and Outputs

- If models A and B take different inputs, handle that in forward().
- If they output different shapes, you may need to add a projection layer (e.g., nn.Linear) to align sizes before fusion.

**Tips**
- Use pretrained encoders from Hugging Face, torchvision, etc.
- Add dropout, batch norm, or attention between the fusion and classifier if performance needs tuning.
- You can use learned fusion methods (e.g., gated fusion, transformer attention between outputs) for better performance.